# 02 — Le cycle complet sur l'exemple 01

Ce notebook déroule pas à pas ce que fait `examples/01_text_classification.py` : créer le projet, importer les assets, envoyer des prédictions, exporter les annotations.

Le script reste la référence ; ici on s'arrête sur chaque étape pour regarder les données qui circulent.

## 0. Prérequis

Générer les données d'exemple si ce n'est pas déjà fait :

```bash
uv run python scripts/generate_sample_data.py
```

In [ ]:
import importlib.util
import json
from pathlib import Path

# On importe le script d'exemple pour réutiliser ses fonctions,
# plutôt que de recopier son code ici.
chemin = Path.cwd().parent / "examples" / "01_text_classification.py"
spec = importlib.util.spec_from_file_location("ex01", chemin)
ex01 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ex01)
print(
    "Fonctions disponibles :",
    [
        n
        for n in dir(ex01)
        if not n.startswith("_") and callable(getattr(ex01, n))
    ],
)

## 1. Les assets

Pour un projet `TEXT`, le contenu de l'asset **est** le texte : pas de fichier à téléverser.

In [ ]:
declarations = ex01.load_declarations()
print(f"{len(declarations)} déclarations")
print(json.dumps(declarations[0], indent=2, ensure_ascii=False))

## 2. L'interface

Un job principal, un sous-job conditionnel.

In [ ]:
json_interface = ex01.build_interface()
print(json.dumps(json_interface, indent=2, ensure_ascii=False))

## 3. Les prédictions

`predict()` renvoie un `json_response` : un dictionnaire dont les clés sont des **noms de jobs**. Regardons ce que donne une déclaration classée « auto » — celle-ci active le sous-job.

In [ ]:
exemple_auto = {
    "external_id": "demo",
    "text": "Collision avec un autre véhicule au rond-point.",
}
print(json.dumps(ex01.predict(exemple_auto), indent=2, ensure_ascii=False))

Observez la clé `children` **à l'intérieur de la catégorie** `SINISTRE_AUTO` : c'est ainsi qu'on répond à un sous-job de classification.

Comparons avec un dégât des eaux, qui n'ouvre pas la branche auto :

In [ ]:
exemple_eaux = {
    "external_id": "demo2",
    "text": "Une infiltration d'eau a endommagé le plafond.",
}
print(json.dumps(ex01.predict(exemple_eaux), indent=2, ensure_ascii=False))

## 4. Vérifier la cohérence hors ligne

Le SDK sait valider un `json_response` contre un `json_interface`, **sans appel réseau**. C'est le meilleur filet de sécurité avant d'envoyer quoi que ce soit à l'instance.

In [ ]:
from kili.services.label_data_parsing.json_response import ParsedJobs
from kili.services.label_data_parsing.types import Project

parsed = ParsedJobs(
    json_response=ex01.predict(exemple_auto),
    project_info=Project(
        jsonInterface=json_interface["jobs"],
        inputType="TEXT",
    ),
)
print("Réponse valide pour cette interface.")
print(parsed["CLASSIFICATION_SINISTRE"].category.name)

## 5. Exécution réelle

⚠️ **Les cellules suivantes écrivent sur l'instance Kili.**

Étape 1 — créer le projet :

In [ ]:
from kili_examples.client import get_kili

kili = get_kili()
projet = kili.create_project(
    title=ex01.PROJECT_TITLE,
    description="Cycle complet depuis le notebook.",
    input_type="TEXT",
    json_interface=json_interface,
)
project_id = projet["id"]
print("project_id =", project_id)

Étape 2 — importer les assets :

In [ ]:
ex01.upload_assets(kili, project_id)

Étape 3 — importer les prédictions. Elles arrivent comme **pré-annotations** (labels de type `PREDICTION`) que l'annotateur corrigera.

In [ ]:
ex01.upload_predictions(kili, project_id)

Étape 4 — exporter. À ce stade, l'export ne contiendra que les prédictions tant que personne n'a annoté dans l'interface.

In [ ]:
from kili_examples.exports import export_labels_to_json

labels = export_labels_to_json(kili, project_id, Path(ex01.EXPORT_PATH))
print(f"{len(labels)} labels exportés")

**Suite** : `03_interface_personnalisee.ipynb` construit pas à pas l'interface multi-jobs de l'exemple 09.